# Notebook 02: LSTM Action Recognition Model Training
## Two-Stage Human Action Recognition System (Fall Detection)

This notebook trains the **LSTM Baseline** action classifier across 8 action classes:
`Walk`, `Eat`, `Drink`, `Fall`, `Lying`, `Sit_Down`, `Stand_Up`, `Crouching`.

### Key Methodological Guarantees:
1. **Leak-Free Video-Level Split**: The 559 valid video clips were split **at the clip level** (`393` Train / `83` Val / `83` Test, stratified, `SEED=42`) *before* sliding-window slicing (`WINDOW_SIZE=16`, `stride=4`).
2. **Minority-Only Train Augmentation**: Only the 3 short-clip minority classes (`Sit_Down`: +4 copies, `Stand_Up`: +4 copies, `Crouching`: +3 copies; rotation `+-10 deg`) receive synthetic copies in Train. The 5 major classes stay at `1x` base count. Train imbalance drops from **`9.00x` (`468/52`) to `1.80x` (`468/260`)**. Val (`524`) and Test (`506`) are 100% un-augmented.
3. **Frozen Safety-Prioritized Selection on Validation**: Uses `Safety Score = 0.50 * Fall_Class_Recall + 0.25 * Fall_Class_F1 + 0.25 * Macro_F1` on Validation.
4. **Dual-Cadence Video-Level Stream Replay (`10.0 FPS` Target vs. `8.33 FPS` CPU-Realistic)**: Streams each of the 83 held-out Test videos through a fresh `SequenceBuffer(16)` + fresh `AlertStateMachine` per video, reporting:
   - **Rule 1**: `Video-Level Fall Clip Recall` (`TP/13`) & `Video-Level False Positive Rate` (`FP/70`)
   - **Rule 2**: `Video-Level RED Trigger Recall` (`TP/13`), `Video-Level RED False Positive Rate` (`FP/70`), and measured confirmation delay (`s`).

## 1. Environment & Reproducibility Setup

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Sanity Check — Overfitting on Toy Subset

In [ ]:
toy_x_path = Path('data/skeleton_dataset/toy/X_toy.npy')
toy_y_path = Path('data/skeleton_dataset/toy/y_toy.npy')

if toy_x_path.exists() and toy_y_path.exists():
    X_toy = np.load(toy_x_path)
    y_toy = np.load(toy_y_path)
    N, T, J, C = X_toy.shape
    X_toy_flat = X_toy.reshape(N, T, J * C).astype(np.float32)
    toy_ds = TensorDataset(torch.from_numpy(X_toy_flat), torch.from_numpy(y_toy).long())
    toy_loader = DataLoader(toy_ds, batch_size=32, shuffle=True)
    
    class ToyLSTM(nn.Module):
        def __init__(self):
            super().__init__()
            self.lstm = nn.LSTM(51, 64, num_layers=2, batch_first=True, bidirectional=True)
            self.fc = nn.Linear(128, 2)
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :])
            
    toy_model = ToyLSTM().to(device)
    toy_opt = torch.optim.Adam(toy_model.parameters(), lr=1e-3)
    toy_crit = nn.CrossEntropyLoss()
    for epoch in range(1, 101):
        toy_model.train()
        for xb, yb in toy_loader:
            xb, yb = xb.to(device), yb.to(device)
            toy_opt.zero_grad()
            loss = toy_crit(toy_model(xb), yb)
            loss.backward()
            toy_opt.step()
    toy_model.eval()
    with torch.no_grad():
        preds = toy_model(torch.from_numpy(X_toy_flat).to(device)).argmax(1).cpu().numpy()
        acc = (preds == y_toy).mean() * 100
    print(f'Sanity Check Overfit Accuracy: {acc:.1f}% (Loss: {loss.item():.4f})')
else:
    print('Toy dataset not attached (running on Kaggle); proceeding to full dataset.')

## 3. Load Leak-Free Video-Level Split Dataset

In [ ]:
# Locate leak-free video-level split files (local or Kaggle)
DATA_DIR = Path('data/skeleton_dataset/windowed')
if not (DATA_DIR / 'X_train.npy').exists() and Path('/kaggle/input').exists():
    candidates = list(Path('/kaggle/input').glob('**/X_train.npy'))
    if candidates:
        DATA_DIR = candidates[0].parent

print(f'Loading leak-free splits from: {DATA_DIR}')
X_train_raw = np.load(DATA_DIR / 'X_train.npy')  # (3175, 16, 17, 3) — 2414 base + 761 minority aug
y_train     = np.load(DATA_DIR / 'y_train.npy')
X_val_raw   = np.load(DATA_DIR / 'X_val.npy')    # (524, 16, 17, 3)  — pure unseen clips @ exact 10.0 FPS
y_val       = np.load(DATA_DIR / 'y_val.npy')
X_test_raw  = np.load(DATA_DIR / 'X_test.npy')   # (506, 16, 17, 3)  — pure unseen clips @ exact 10.0 FPS
y_test      = np.load(DATA_DIR / 'y_test.npy')

cids_val  = np.load(DATA_DIR / 'clip_ids_val.npy')  if (DATA_DIR / 'clip_ids_val.npy').exists()  else None
cids_test = np.load(DATA_DIR / 'clip_ids_test.npy') if (DATA_DIR / 'clip_ids_test.npy').exists() else None
test_seq_path = DATA_DIR / 'test_sequences.npz'

# Verify split manifest (zero clip overlap across Train / Val / Test)
manifest_candidates = [
    DATA_DIR / 'split_manifest.json',
    DATA_DIR.parent / 'split_manifest.json',
]
if Path('/kaggle/input').exists():
    manifest_candidates.extend(Path('/kaggle/input').glob('**/split_manifest.json'))
for mp in manifest_candidates:
    if mp.exists():
        with open(mp, 'r', encoding='utf-8') as f:
            manifest = json.load(f)
        for cls, sp in manifest['splits_by_class'].items():
            assert not (set(sp['train_clips']) & set(sp['val_clips'])), f'Leakage train/val in {cls}'
            assert not (set(sp['train_clips']) & set(sp['test_clips'])), f'Leakage train/test in {cls}'
            assert not (set(sp['val_clips']) & set(sp['test_clips'])), f'Leakage val/test in {cls}'
        print('Verified split_manifest.json: ZERO video clip overlap across Train, Val, and Test!')
        print('Clip counts:', manifest['clip_counts'])
        break

CLASS_NAMES = ['Walk', 'Eat', 'Drink', 'Fall', 'Lying', 'Sit_Down', 'Stand_Up', 'Crouching']
NUM_CLASSES = len(CLASS_NAMES)
LABEL_TO_ALERT = {
    'Walk': 'GREEN', 'Eat': 'GREEN', 'Drink': 'GREEN', 'Sit_Down': 'GREEN', 'Stand_Up': 'GREEN',
    'Crouching': 'YELLOW', 'Lying': 'YELLOW',
    'Fall': 'RED',
}

# Self-verifying class distribution & imbalance ratio from y_train
classes, counts = np.unique(y_train, return_counts=True)
imbalance_ratio = counts.max() / counts.min()
print(f'\nWindow shapes: Train={X_train_raw.shape}, Val={X_val_raw.shape}, Test={X_test_raw.shape}')
print(f'{"Class":<12} | {"Train":>7} | {"Val (Pure)":>10} | {"Test (Pure)":>11}')
print('-' * 50)
for cls_idx, count in zip(classes, counts):
    name = CLASS_NAMES[cls_idx]
    print(f'{name:<12} | {count:7d} | {(y_val == cls_idx).sum():10d} | {(y_test == cls_idx).sum():11d}')
print('-' * 50)
print(f'Train Post-Augmentation Imbalance Ratio: {counts.max()}/{counts.min()} = {imbalance_ratio:.2f}x')

## 4. Prepare Flattened Tensors & Post-Augmentation Class Weights

In [ ]:
# Flatten per-frame keypoints for LSTM: (N, 16, 17, 3) -> (N, 16, 51)
X_train = X_train_raw.reshape(len(X_train_raw), 16, 51).astype(np.float32)
X_val   = X_val_raw.reshape(len(X_val_raw), 16, 51).astype(np.float32)
X_test  = X_test_raw.reshape(len(X_test_raw), 16, 51).astype(np.float32)

# Compute class weights strictly on the POST-AUGMENTATION y_train distribution
class_counts = np.bincount(y_train, minlength=NUM_CLASSES)
total_train = len(y_train)
class_weights = total_train / (NUM_CLASSES * class_counts.astype(np.float32))
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print('Post-augmentation class weights for CrossEntropyLoss (max/min ratio = 1.80x):')
for name, cnt, w in zip(CLASS_NAMES, class_counts, class_weights):
    print(f'  {name:10s}: count={cnt:4d} -> weight={w:.3f}')

## 5. Bi-LSTM Model Architecture (Matches `src/pipeline/action_classifier.py`)

In [ ]:
class ActionLSTM(nn.Module):
    def __init__(
        self,
        input_size: int = 51,
        hidden_size: int = 128,
        num_layers: int = 2,
        num_classes: int = 8,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)     # (B, T, 2 * hidden_size)
        out = out[:, -1, :]       # Last time step
        out = self.dropout(out)
        return self.fc(out)       # (B, num_classes)

## 6. Train & Select on Validation (Frozen Safety Score: Unweighted vs. Weighted Loss)

In [ ]:
def compute_safety_metrics(y_true, y_pred, fall_idx=3):
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    fall_mask = (y_true == fall_idx)
    pred_fall = (y_pred == fall_idx)
    tp = int((fall_mask & pred_fall).sum())
    fp = int((~fall_mask & pred_fall).sum())
    fn = int((fall_mask & ~pred_fall).sum())
    fall_class_rec  = tp / max(1, tp + fn)
    fall_class_prec = tp / max(1, tp + fp)
    fall_class_f1   = 2 * fall_class_rec * fall_class_prec / max(1e-8, fall_class_rec + fall_class_prec)
    # Frozen Safety Score: 50% Fall Class Recall + 25% Fall Class F1 + 25% Macro-F1
    safety_score = 0.50 * fall_class_rec + 0.25 * fall_class_f1 + 0.25 * macro_f1
    return {
        'safety_score': safety_score,
        'fall_class_recall': fall_class_rec * 100.0,
        'fall_class_precision': fall_class_prec * 100.0,
        'fall_class_f1': fall_class_f1 * 100.0,
        'macro_f1': macro_f1,
        'acc': float((y_true == y_pred).mean() * 100.0),
    }

BATCH_SIZE = 64
EPOCHS = 45
LR = 1e-3

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).long()),
    batch_size=BATCH_SIZE, shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val).long()),
    batch_size=BATCH_SIZE, shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test).long()),
    batch_size=BATCH_SIZE, shuffle=False,
)

def train_lstm_variant(use_class_weights: bool):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    m = ActionLSTM(input_size=51, hidden_size=128, num_layers=2, num_classes=NUM_CLASSES).to(device)
    crit = nn.CrossEntropyLoss(weight=weights_tensor if use_class_weights else None)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    best_score = -1.0
    best_state = None
    best_metrics = {}

    tag = 'Weighted' if use_class_weights else 'Unweighted'
    print(f'\n--- Training LSTM ({tag} Loss) ---')
    for epoch in range(1, EPOCHS + 1):
        m.train()
        tr_loss, n_tr = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = m(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * len(xb)
            n_tr += len(xb)
        sched.step()

        m.eval()
        v_preds, v_targets = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = m(xb.to(device))
                v_preds.extend(logits.argmax(1).cpu().numpy())
                v_targets.extend(yb.numpy())
        metrics = compute_safety_metrics(np.array(v_targets), np.array(v_preds))
        metrics['epoch'] = epoch

        if metrics['safety_score'] > best_score:
            best_score = metrics['safety_score']
            best_state = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            best_metrics = metrics

        if epoch % 10 == 0 or epoch == 1 or epoch == EPOCHS:
            print(f'  Epoch {epoch:2d}/{EPOCHS} | TrainLoss={tr_loss/n_tr:.4f} | ValFallClassRec={metrics["fall_class_recall"]:.1f}% | ValFallPrec={metrics["fall_class_precision"]:.1f}% | ValMacroF1={metrics["macro_f1"]:.4f} | SafetyScore={metrics["safety_score"]:.4f}')
    return best_state, best_metrics

state_unw, metrics_unw = train_lstm_variant(use_class_weights=False)
state_w,   metrics_w   = train_lstm_variant(use_class_weights=True)

print('\n' + '=' * 76)
print('Validation Model Selection (Safety Score = 0.50*FallClassRec + 0.25*FallF1 + 0.25*MacroF1):')
print(f'  Unweighted Loss : SafetyScore={metrics_unw["safety_score"]:.4f} | FallClassRec={metrics_unw["fall_class_recall"]:.2f}% | FallPrec={metrics_unw["fall_class_precision"]:.2f}% | MacroF1={metrics_unw["macro_f1"]:.4f} (ep {metrics_unw["epoch"]})')
print(f'  Weighted Loss   : SafetyScore={metrics_w["safety_score"]:.4f} | FallClassRec={metrics_w["fall_class_recall"]:.2f}% | FallPrec={metrics_w["fall_class_precision"]:.2f}% | MacroF1={metrics_w["macro_f1"]:.4f} (ep {metrics_w["epoch"]})')
print('=' * 76)

if metrics_w['safety_score'] >= metrics_unw['safety_score']:
    best_weights = state_w
    print('-> Selected on Validation: Post-Augmentation Weighted Loss checkpoint')
else:
    best_weights = state_unw
    print('-> Selected on Validation: Unweighted Loss checkpoint')

model = ActionLSTM(input_size=51, hidden_size=128, num_layers=2, num_classes=NUM_CLASSES).to(device)
model.load_state_dict(best_weights)

## 7. Held-Out Test Set Evaluation (Window-Level + Video-Level + Dual-Cadence Stream Replay)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import collections
import time

model.eval()
all_preds = []
all_probs = []
all_targets = []

# Latency benchmark (single sequence)
latencies = []
sample_input = torch.from_numpy(X_test[:1]).to(device)
with torch.no_grad():
    for _ in range(10):
        _ = model(sample_input)
    for _ in range(100):
        t0 = time.perf_counter()
        _ = model(sample_input)
        latencies.append((time.perf_counter() - t0) * 1000.0)

    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_targets.extend(yb.numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
fall_idx = CLASS_NAMES.index('Fall')

print(f'Mean inference latency: {np.mean(latencies):.3f} ms +/- {np.std(latencies):.3f} ms per sequence')
print('\n' + '=' * 76)
print('1. WINDOW-LEVEL Classification Report (506 Windows, stride=4 @ exact 10.0 FPS):')
print('=' * 76)
print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES, digits=4))

cm8 = confusion_matrix(all_targets, all_preds)
print('8x8 Window-Level Confusion Matrix (rows=true, cols=pred):')
print(cm8)

# 2A. Video-Level Classification Report (Mean-Probability over windows of each clip)
if cids_test is not None and len(cids_test) == len(all_targets):
    unique_clips = sorted(set(cids_test.tolist()))
    clip_true, clip_pred_mean = [], []
    for cid in unique_clips:
        mask = (cids_test == cid)
        clip_true.append(int(all_targets[mask][0]))
        clip_pred_mean.append(int(np.argmax(all_probs[mask].mean(axis=0))))
    clip_true = np.array(clip_true)
    clip_pred_mean = np.array(clip_pred_mean)
    print('\n' + '=' * 76)
    print(f'2A. VIDEO-LEVEL Classification Report — Mean Prob ({len(unique_clips)} Independent Clips):')
    print('=' * 76)
    print(classification_report(clip_true, clip_pred_mean, target_names=CLASS_NAMES, digits=4))

# 2B. STREAM-CADENCE VIDEO-LEVEL TRIGGER EVALUATION (Target 10.0 FPS vs. CPU-Realistic 8.33 FPS)
# Note: Each sequence in `test_sequences.npz` was already resampled to 10.0 FPS during extraction.
#   • Target-Cadence Replay (10.0 FPS, 0.10s/tick): steps through every 10-FPS frame (stride=1).
#   • CPU-Realistic Replay (8.33 FPS, 0.12s/tick): simulates frame-skipping at 8.33 FPS
#     (per spec.md Section 3.2 frame-dropping governance).
if test_seq_path.exists():
    test_seqs = np.load(test_seq_path)

    def run_stream_replay(replay_fps: float):
        tp_r1, fp_r1 = 0, 0
        tp_r2, fp_r2 = 0, 0
        n_fall_vids, n_nonfall_vids = 0, 0
        trigger_delays_sec = []

        with torch.no_grad():
            for clip_key in sorted(test_seqs.files):
                cls_name = clip_key.split('/')[0]
                true_is_fall = (cls_name == 'Fall')
                seq_10fps = test_seqs[clip_key]  # (T_10fps, 17, 3) @ exact 10.0 FPS (0.10s spacing)
                duration = len(seq_10fps) / 10.0
                # Timestamp-based resampling from 10.0 FPS source to replay_fps
                target_times = np.arange(0.0, duration, 1.0 / replay_fps, dtype=np.float64)
                idx_arr = np.clip(np.floor(target_times * 10.0 + 0.5).astype(int), 0, max(0, len(seq_10fps) - 1))
                seq = seq_10fps[idx_arr]
                # Actual source timestamps (seconds) of the selected frames
                frame_times = idx_arr.astype(np.float64) * 0.10
                if len(seq) < 16:
                    pad_n = 16 - len(seq)
                    pad = np.tile(seq[-1:], (pad_n, 1, 1))
                    seq = np.concatenate([seq, pad], axis=0)
                    pad_times = frame_times[-1] + np.arange(1, pad_n + 1, dtype=np.float64) / replay_fps
                    frame_times = np.concatenate([frame_times, pad_times], axis=0)

                # Fresh SequenceBuffer & fresh AlertStateMachine counters per video
                buf = collections.deque(maxlen=16)
                valid_flags = collections.deque(maxlen=16)
                consec_missing = 0
                crit_count = 0
                red_triggered = False
                any_fall_tick = False
                first_crit_time = None

                for tick_idx, (frame_kp, t_sec) in enumerate(zip(seq, frame_times)):
                    is_zero = bool(np.all(frame_kp == 0.0))
                    if is_zero:
                        consec_missing += 1
                        if consec_missing > 3:
                            buf.clear()
                            valid_flags.clear()
                            crit_count = 0
                            first_crit_time = None
                            continue
                        buf.append(np.zeros((17, 3), dtype=np.float32))
                        valid_flags.append(False)
                    else:
                        consec_missing = 0
                        buf.append(frame_kp.astype(np.float32))
                        valid_flags.append(True)

                    if len(buf) < 16:
                        continue
                    if sum(valid_flags) < 8:
                        crit_count = 0
                        first_crit_time = None
                        continue

                    win = np.stack(list(buf), axis=0)
                    win_t = torch.from_numpy(win.reshape(1, 16, 51).astype(np.float32)).to(device)
                    probs_t = torch.softmax(model(win_t), dim=-1)[0].cpu().numpy()
                    pred_idx = int(np.argmax(probs_t))
                    pred_conf = float(probs_t[pred_idx])
                    pred_label = CLASS_NAMES[pred_idx]

                    if pred_conf < 0.50:
                        crit_count = 0
                        first_crit_time = None
                        continue

                    if pred_label == 'Fall':
                        any_fall_tick = True
                        if pred_conf >= 0.90:
                            if not red_triggered:
                                red_triggered = True
                                if true_is_fall:
                                    delay_s = 0.0 if first_crit_time is None else float(t_sec - first_crit_time)
                                    trigger_delays_sec.append(delay_s)
                            crit_count = 0
                        elif pred_conf >= 0.75:
                            if crit_count == 0:
                                first_crit_time = float(t_sec)
                            crit_count += 1
                            if crit_count >= 3:
                                if not red_triggered:
                                    red_triggered = True
                                    if true_is_fall and first_crit_time is not None:
                                        trigger_delays_sec.append(float(t_sec - first_crit_time))
                                crit_count = 0
                        else:
                            crit_count = 0
                            first_crit_time = None
                    else:
                        crit_count = 0
                        first_crit_time = None

                if true_is_fall:
                    n_fall_vids += 1
                    if any_fall_tick: tp_r1 += 1
                    if red_triggered: tp_r2 += 1
                else:
                    n_nonfall_vids += 1
                    if any_fall_tick: fp_r1 += 1
                    if red_triggered: fp_r2 += 1

        mean_delay = float(np.mean(trigger_delays_sec)) if trigger_delays_sec else 0.0
        return {
            'n_fall': n_fall_vids, 'n_nonfall': n_nonfall_vids,
            'tp_r1': tp_r1, 'fp_r1': fp_r1,
            'tp_r2': tp_r2, 'fp_r2': fp_r2,
            'mean_delay_sec': mean_delay,
        }

    print('\n' + '=' * 76)
    print('2B. STREAM-CADENCE VIDEO-LEVEL TRIGGER EVALUATION (13 Fall vs. 70 Non-Fall Videos):')
    print('=' * 76)
    for mode_label, fps_val in [('Target-Cadence Replay (10.0 FPS design)', 10.0), ('CPU-Realistic Replay (8.33 FPS measured)', 8.33)]:
        res = run_stream_replay(fps_val)
        nf, nnf = res['n_fall'], res['n_nonfall']
        print(f'\n  >>> {mode_label} <<<')
        print(f'  [Rule 1: >=1 Fall Tick]          Video-Level Fall Clip Recall      : {res["tp_r1"]:2d}/{nf} ({100*res["tp_r1"]/max(1,nf):5.2f}%)')
        print(f'                                   Video-Level False Positive Rate   : {res["fp_r1"]:2d}/{nnf} ({100*res["fp_r1"]/max(1,nnf):5.2f}%)')
        print(f'  [Rule 2: AlertStateMachine RED]  Video-Level RED Trigger Recall    : {res["tp_r2"]:2d}/{nf} ({100*res["tp_r2"]/max(1,nf):5.2f}%)')
        print(f'                                   Video-Level RED False Positive Rate: {res["fp_r2"]:2d}/{nnf} ({100*res["fp_r2"]/max(1,nnf):5.2f}%)')
        print(f'                                   Mean Confirmation Delay (1st->RED): {res["mean_delay_sec"]:.3f} s')

# 3. Window-level Fall Class Recall & Postural Confusion Analysis
fall_mask = (all_targets == fall_idx)
fall_total = int(fall_mask.sum())
fall_tp = int((fall_mask & (all_preds == fall_idx)).sum())
fall_fn = fall_total - fall_tp
fall_class_recall = fall_tp / fall_total if fall_total > 0 else 0.0
fall_fnr = fall_fn / fall_total if fall_total > 0 else 0.0
macro_f1 = f1_score(all_targets, all_preds, average='macro')

print('\n' + '=' * 76)
print('3. Fall Class Recall & Postural Confusion Analysis (Window-Level):')
print('=' * 76)
print(f'  Macro-F1 (8 classes)         : {macro_f1:.4f}')
print(f'  Fall Class Recall            : {fall_class_recall * 100:.2f}% ({fall_tp}/{fall_total})')
print(f'  Fall False Negative Rate     : {fall_fnr * 100:.2f}% ({fall_fn}/{fall_total})')
for conf_cls in ['Lying', 'Crouching', 'Sit_Down']:
    c_idx = CLASS_NAMES.index(conf_cls)
    n_fall_as_c = int((fall_mask & (all_preds == c_idx)).sum())
    n_c_as_fall = int(((all_targets == c_idx) & (all_preds == fall_idx)).sum())
    print(f'  Fall misclassified as {conf_cls:<9}: {n_fall_as_c:3d} | {conf_cls:<9} misclassified as Fall: {n_c_as_fall:3d}')

# 4. 3-Level Alert Mapping Evaluation (GREEN / YELLOW / RED)
ALERT_LEVELS = ['GREEN', 'YELLOW', 'RED']
ALERT_TO_IDX = {'GREEN': 0, 'YELLOW': 1, 'RED': 2}
y_true_alert = np.array([ALERT_TO_IDX[LABEL_TO_ALERT[CLASS_NAMES[i]]] for i in all_targets])
y_pred_alert = np.array([ALERT_TO_IDX[LABEL_TO_ALERT[CLASS_NAMES[i]]] for i in all_preds])

print('\n' + '=' * 76)
print('4. 3-Level Alert Mapping Report (Window-Level GREEN / YELLOW / RED):')
print('=' * 76)
print(classification_report(y_true_alert, y_pred_alert, target_names=ALERT_LEVELS, digits=4))
print('3x3 Alert Confusion Matrix (rows=true, cols=pred):')
print(confusion_matrix(y_true_alert, y_pred_alert))

## 8. Export Final Model Checkpoint

In [ ]:
out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('models')
out_dir.mkdir(exist_ok=True)
save_path = out_dir / 'action_lstm.pth'
torch.save(best_weights, str(save_path))
print(f'Successfully saved final weights to: {save_path}')